# Data Generation for Grocery Supply Chain

* **General Description**:  
This notebook executes a high-performance data engineering pipeline using Polars to generate synthetic, integrated data for a grocery supply chain. It combines product catalogs, supplier networks, transport route simulations, real INMET weather historical records, seasonal patterns, and simulated sales volumes[cite: 1].


* **Key Pipeline Stages**:
    * **Catalog Loading**: Importing JSON files containing product, category, and supplier information, generating unique identifiers, and enriching the dataset with randomized supplier ratings[cite: 1].


    * **Route Simulation**: Segmenting logistics routes into urban, highway, and off-road portions based on product categories, geographical constraints, and total transit distances[cite: 1].


    * **Weather Data Integration**: Parsing, cleaning, and validating real weather station records (precipitation, temperature, and wind metrics), handling missing values through imputation, and classifying weather severity levels[cite: 1].


    * **Demand and Inventory Modeling**: Merging weather parameters with seasonality rules, calculating business days and Brazilian holidays, simulating daily sales volumes, estimating transit times, applying minimum/maximum inventory policies, and generating automated purchase orders[cite: 1].


* **Output Structure (`final_df`)**: The resulting DataFrame consolidates 31 standardized columns, including purchase and delivery timestamps, product details, supplier attributes, inventory performance metrics, lead times, and classified weather features[cite: 1].

### Import Necessary Libraries

In [67]:
import pandas as pd
import numpy as np
import polars as pl
import polars.selectors as cs
import os
import csv
import json
import fastparquet

import create_data_functions, weather_conditions

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

pl.set_random_seed(seed=56)

In [68]:
import time

start = time.perf_counter()

start_CPU = time.process_time()

### Directory Paths Configuration

In [69]:
# Establish standard directory paths for data ingestion and storage
p = Path("data")

raw_data_path = p / 'raw'

processed_path = p / 'processed'

external_data_path = p / 'external'

# Define dataframe height
length_df = 1_000_000

In [70]:
# Define the list of core JSON catalog files to ingest
arch_json = ['products','products_categories', 'suppliers']

# Initialize a dictionary container to hold the raw JSON datasets in memory
store_catalog = {}

# Iterate through target filenames, resolve file paths, and safely load JSON content
for name in arch_json:
    file_path = raw_data_path / f"{name}.json"  # Construct full file path
    if file_path.exists():
        with file_path.open("r", encoding="utf-8") as f:     # Open the JSON file with UTF-8 encoding
            store_catalog[name] = json.load(f)               # Load and map data to its dictionary key
    else:
        print(f"Notice: File {file_path} was not found.")

# Catalog Information Processing

In [71]:
# Calculate total product count from the loaded catalog dictionary
total_products = len(store_catalog["products"])
# Calculate total supplier count from the loaded catalog dictionary
total_suppliers = len(store_catalog["suppliers"])

# Generate unique identifiers for each supplier using an 'S' prefix
suppliers_id = create_data_functions.create_IDs(total_suppliers, suffix='S')

# Initialize a pseudo-random number generator with a deterministic seed for reproducibility
rng = np.random.default_rng(seed=43)

# Randomly sample 15 unique supplier keys to designate as high-tier preferred partners
suppliers_top = rng.choice(list(set(store_catalog['suppliers'].keys())), 15, replace=False)

# Build an optimized Polars LazyFrame to process product attributes and join supplier details
catalog_lazy = (
    pl.DataFrame(store_catalog["products"])  # Convert raw products dictionary into a DataFrame
    .transpose(include_header=True)          # Transpose data structure to align product records
    .lazy()                                  # Convert to LazyFrame to leverage deferred execution optimization
    .rename({"column": "product"})           # Normalize default header to 'product'
    .unnest("column_0")                      # Expand nested attribute structures
    .with_columns(
        pl.Series(
            "product_id",
            create_data_functions.create_IDs(total_products, suffix='P')  # Assign unique product identifiers ('P' prefix)
        ))
    .select([                                # Filter and order essential product attributes
        "product_id",
        "product",
        "category",
        "sub_category",
        "shelf_life_days",
        "maximum_days_on_sale",
        "seasonality",
        "storage_recommendation",
        "unit_of_measurement"
        ])

    # Perform an inner join to merge product specifications with respective supplier metadata
    .join(
        pl.from_dicts(store_catalog["suppliers"])   # Convert suppliers dictionary collection to a DataFrame
        .transpose(include_header=True)             # Transpose rows and columns for uniform alignment
        .unnest("column_0")                         # Unnest nested dictionary fields
        .with_columns([
            pl.Series("supplier_id", suppliers_id), # Inject generated supplier ID series
            pl.Series("supplier_rating", np.random.randint(1, 6, size=total_suppliers)).cast(pl.UInt8)  # Assign random supplier performance ratings (1-5)
        ])
        .lazy()                                     # Elevate supplier DataFrame to a LazyFrame
        .rename({
            "products": "product",                  # Map supplier product lists column to 'product'
            "column": "supplier"                    # Rename structural header column to 'supplier'
        })
        .explode("product")                         # Explode item lists to create a granular product-supplier mapping
        .select([                                   # Select relevant metrics from the supplier dataset
            "supplier_id", "supplier_rating", "supplier", "product", "distance_km", "moq"
        ])
    ,
    on="product",                                   # Join dataframes cleanly on the common product field
    how="inner"                                     # Execute inner join to retain only matched records
    )
)

## Simulating Lead Time with Urban, Highway, and Off-Road Segments

This section models logistics delivery lead times by breaking down each supplier transit route into three distinct segments:

- **Urban segment**: Always allocates the initial 50 km of transit, capturing local metropolitan distribution and urban traffic constraints.
- **Highway segment**: Captures the remaining trunk distance after the urban portion, typically utilized for moving industrial or processed merchandise.
- **Off-road segment**: A localized stochastic fraction of the remaining distance applied explicitly to agricultural or fresh food categories requiring rural facility access (e.g., Meat, Seafood, Vegetables, Fruits, Dairy, Eggs, Grains & Rice, Dried Fruits).

### Simulation Logic Workflow
1. **Urban distance** = `min(distance_km, 50)`
2. **Remaining distance** = `max(distance_km - 50, 0)`
3. For categories requiring rural accessibility:
   - Designate a random proportion spanning 10% to 40% of the remaining distance as off-road.
   - Allocate the balance to the highway segment.
4. For standard/processed product categories:
   - Off-road distance = 0
   - Highway distance = full remaining distance

This methodology guarantees realistic supply chain variance:
- Short delivery routes (≤ 50 km) remain exclusively urban.
- Extended routes for fresh produce account for rural terrain access.
- Processed items rely completely on standard highway and urban infrastructures.

The resulting enriched dataset outputs three distinct features:
- `urban_km`
- `highway_km`
- `off_road_km`

These features downstream model granular transit durations, fuel consumption costs, and delivery vulnerability profiles based on specific route topographies.


In [72]:
# Execute the custom road network simulation function to enrich catalog logistics features 
catalog_lazy = create_data_functions.road_simulation_polars(catalog_lazy)

## Meteorological Data Processing for Supply Chain Context

In [73]:
# Define the absolute path to external weather station data archives
# Reference Source: National Institute of Meteorology (INMET) - https://bdmep.inmet.gov.br/[cite: 1]

# Set target path for the historical weather CSV input file
archive_csv = external_data_path / 'dados_B807_D_2022-12-07_2025-09-22.csv'

# Map original Portuguese weather dataset headers to clean, standardized English variable names
columns_name = {
    "Data Medicao": "measurement_date",
    "PRECIPITACAO TOTAL, DIARIO (AUT)(mm)": "daily_total_precipitation_mm",
    "TEMPERATURA MAXIMA, DIARIA (AUT)(°C)": "daily_maximum_temperature_c",
    "TEMPERATURA MINIMA, DIARIA (AUT)(°C)": "daily_minimum_temperature_c",
    "VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)": "daily_average_wind_speed_mps"
}

In [74]:
# Stream and process weather CSV records via a Polars LazyFrame for optimized query planning
weather_lazy = (
    pl.scan_csv(
        archive_csv,
        separator=";",          # Specify semicolon as the file column separator
        decimal_comma=True,     # Instruct parser to interpret commas correctly as decimal marks
        skip_rows=10,           # Bypass metadata header rows at the file summit
        try_parse_dates=True    # Enable intelligent automatic date inference
    )
    # Apply column renaming dictionary for clear English identification
    .rename(columns_name)

    # Drop any phantom empty columns generated by trailing delimiters in source CSV rows
    .drop("")

    # Standardize string representations of nulls into native missing values and cast columns to Float32
    .with_columns([
        pl.when(pl.col(c) == "null")
        .then(None)             # Translate explicit string "null" tokens to null objects
        .otherwise(pl.col(c))   # Preserve valid recorded values intact
        .cast(pl.Float32)       # Convert meteorological metrics to standard 32-bit floats
        .alias(c)               # Maintain original field naming convention
        for c in columns_name.values() if c != "measurement_date"
    ])

    # Filter out completely blank records where all measured values register as null
    .filter(~pl.all_horizontal(pl.all().is_null()))

    # Strip initial artifact rows to finalize tabular alignment
    .slice(2, None)

    .with_columns(
        # Generate tracking indicator flags for missing entries prior to imputation
        pl.col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"),
        pl.col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"),
        pl.col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"),
        pl.col("daily_average_wind_speed_mps").is_null().alias("wind_missing"),

        # Apply forward-fill imputation strategy to propagate preceding valid readings forward
        pl.col("daily_total_precipitation_mm").fill_null(strategy="forward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="forward"),
    )

    .with_columns(
        # Apply backward-fill imputation strategy to resolve remaining missing values from subsequent records
        pl.col("daily_total_precipitation_mm").fill_null(strategy="backward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="backward"),
    )

    # Select final organized column projection combining core metrics and missingness tracking flags
    .select(
        [
            'measurement_date',                # Primary chronological identifier
            'daily_total_precipitation_mm',    # Cleaned precipitation measurements
            'precipitation_missing',           # Binary indicator flag for imputed precipitation records
            'daily_maximum_temperature_c',     # Cleaned maximum temperature measurements
            'max_temp_missing',                # Binary indicator flag for imputed max temperatures
            'daily_minimum_temperature_c',     # Cleaned minimum temperature measurements
            'min_temp_missing',                # Binary indicator flag for imputed min temperatures
            'daily_average_wind_speed_mps',    # Cleaned average wind speed measurements
            'wind_missing'                     # Binary indicator flag for imputed wind speeds
        ]
    ) 
)

In [75]:
# Execute and materialize the weather LazyFrame into memory for structural verification
df_weather = weather_lazy.collect()

# Extract the chronological starting date of the recording period
date_min = df_weather["measurement_date"].min()

# Extract the chronological ending date of the recording period
date_max = df_weather["measurement_date"].max()

# Construct a continuous daily reference date sequence spanning from start to finish
expect_seq = pl.date_ranges(date_min, date_max, interval="1d", eager=True).explode()

# Retrieve distinct active dates present in the dataset and sort them chronologically
exist_date = df_weather["measurement_date"].unique().sort()

# Perform validation checks to detect potential chronological data gaps in the time series
gaps = not exist_date.equals(expect_seq)

# Report whether temporal gaps are present in the weather sequence
print(f"Have date gaps? {gaps}")

Have date gaps? False


In [76]:
# Inspect data schema details lazily without triggering a full materialization pass
print(weather_lazy.collect_schema())

Schema([('measurement_date', Date), ('daily_total_precipitation_mm', Float32), ('precipitation_missing', Boolean), ('daily_maximum_temperature_c', Float32), ('max_temp_missing', Boolean), ('daily_minimum_temperature_c', Float32), ('min_temp_missing', Boolean), ('daily_average_wind_speed_mps', Float32), ('wind_missing', Boolean)])


In [77]:
# Output the optimized execution plan generated by Polars for the weather pipeline
print(weather_lazy.explain())

simple π 9/9 ["measurement_date", ... 8 other columns]
   WITH_COLUMNS:
   [col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
     WITH_COLUMNS:
     [col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"), col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"), col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"), col("daily_average_wind_speed_mps").is_null().alias("wind_missing"), col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
      SLICE[offset: 2, len: 18446744073709551615]
        FILTER [([([([(col("measurement_date").is_not_nu

### Define Weather Severity Levels and Classifications

In [78]:
# Instantiate weather analytics module and apply categorization rules to compute severity features
weather_analyser = weather_conditions.PolarsWeatherConditions(weather_lazy)
weather_severity_lazy = (
    weather_analyser.classify_weather()
    .select([
            'measurement_date',
            'temperature_classification',
            'precipitation_classification',
            'wind_classification', 
            'weather_severity'
        ])
    .rename({"measurement_date": "received_date"})
)

In [79]:
# Preview the first 3 rows of the processed weather severity features
weather_severity_lazy.head(3).collect()

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,str,str,str,str
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""


In [80]:
# Validate and return the schema definition of the weather severity LazyFrame
weather_severity_lazy.collect_schema()

Schema([('received_date', Date),
        ('temperature_classification', String),
        ('precipitation_classification', String),
        ('wind_classification', String),
        ('weather_severity', String)])

# Comprehensive Supply Chain Modeling based on Weather, Product Specifications, and Seasonality

In [81]:
# Compute total sample count based on the length of the weather severity dataset
n_samples = len(weather_severity_lazy.collect())

# Calculate required replication factor to scale dataset row counts to target dimensions (~300,000 rows)
multiply_rows = length_df // n_samples + 1
n_total = multiply_rows * n_samples

# Replicate weather feature rows across the calculated multiplier factor
weather_replicated_lazy = pl.concat([weather_severity_lazy] * multiply_rows)

# Sample product catalog records with replacement to match the desired total sample size
catalog_sampled_lazy = catalog_lazy.collect().sample(n=n_total, with_replacement=True)

# Horizontally concatenate weather conditions and product attributes, sort chronologically, and evaluate seasonality alignment
merged_prod_weather_lazy = (
    weather_replicated_lazy
    .collect()
    .hstack(catalog_sampled_lazy)
    .sort("received_date")
    .with_columns(
        (pl.col("received_date").dt.strftime("%B").is_in(pl.col("seasonality")).alias("in_season"))
    )
).lazy()

In [82]:
# Display a quick inspection preview of the merged production dataset
merged_prod_weather_lazy.show(3)

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,off_road_km,highway_km,urban_km,in_season
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,i32,i64,i64,bool
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1194877|S""",4,"""ValleyFresh Farms""",85,100,5,30,50,false
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""",14,5,[],"""Refrigerated""","""lb""","""1422853|S""",5,"""Artisan Cheesemakers""",95,40,7,38,50,false
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1113278|P""","""Carrot""","""Fresh Foods""","""Vegetables""",21,7,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""","""1194877|S""",4,"""ValleyFresh Farms""",85,100,7,28,50,true


In [83]:
# Print the complete schema of the combined weather and product pipeline frame
print(merged_prod_weather_lazy.collect_schema())

Schema([('received_date', Date), ('temperature_classification', String), ('precipitation_classification', String), ('wind_classification', String), ('weather_severity', String), ('product_id', String), ('product', String), ('category', String), ('sub_category', String), ('shelf_life_days', Int64), ('maximum_days_on_sale', Int64), ('seasonality', List(String)), ('storage_recommendation', String), ('unit_of_measurement', String), ('supplier_id', String), ('supplier_rating', UInt8), ('supplier', String), ('distance_km', Int64), ('moq', Int64), ('off_road_km', Int32), ('highway_km', Int64), ('urban_km', Int64), ('in_season', Boolean)])


## Holiday and Calendar Day Classification

In [84]:
# Classify each record's delivery timestamp into specific calendar day categories (weekday, weekend, or official holiday)
# tailored for Brazil ('br'), returning an enriched lazy DataFrame with temporal context.
prod_seasonality_lazy = (
    create_data_functions.day_classification_lazy(
        df=merged_prod_weather_lazy,
        col="received_date",
        country="br"
    )
)


## Inventory Stock Quantities and Demand Volume Simulation

In [85]:
# Simulate grocery market consumer demand categories based on calendar and regional parameters
data_stock_sales_lazy = create_data_functions.classify_grocery_demand_polars(df=prod_seasonality_lazy, columns_date="received_date", country="br")

# Simulate specific sales volume amounts corresponding to classified demand tiers
data_stock_sales_lazy = create_data_functions.simulate_sales_volume_polars(df= data_stock_sales_lazy)

### Simulating Route Speed Distributions and Total Transit Duration in Minutes

In [86]:
# Model stochastic vehicle speed distributions across urban, highway, and off-road segments to compute total transit times
data_transit_time_lazy = create_data_functions.simulate_distribution_speeds(data_stock_sales_lazy)

### Generate Stock Threshold Parameters and Delivery Estimates

In [87]:
# Estimate granular delivery lead times and decimal delivery days from simulated transit metrics
data_stock_time_lazy = create_data_functions.estimate_delivery_polars(df= data_transit_time_lazy)

# Calculate minimum and maximum stock threshold policies (Min-Max inventory levels)
data_stock_sales_lazy = create_data_functions.min_max_stock_polars(df = data_stock_time_lazy)

### Create Stock Level Distribution

In [88]:
# Generate the inventory stock distribution dataset from processed sales records
# and display a preview to inspect structural integrity.
data_stock_dist_lazy = create_data_functions.create_stock_distribution_polars(data_stock_sales_lazy)

data_stock_dist_lazy.show()

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,in_season,is_holiday,day_classification,is_weekend,sales_demand,sales_volume,transit_time,delivery_days,min_stock,max_stock,stock_quantity
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,bool,bool,str,bool,str,i64,f64,f64,i32,i32,i32
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1194877|S""",4,"""ValleyFresh Farms""",85,100,false,false,"""Weekday""",false,"""High""",164,2.414875,0.605329,269,369,292
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""",14,5,[],"""Refrigerated""","""lb""","""1422853|S""",5,"""Artisan Cheesemakers""",95,40,false,false,"""Weekday""",false,"""High""",108,2.768556,0.931616,237,277,275
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1113278|P""","""Carrot""","""Fresh Foods""","""Vegetables""",21,7,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""","""1194877|S""",4,"""ValleyFresh Farms""",85,100,true,false,"""Weekday""",false,"""High""",163,2.173746,1.413887,269,369,282
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1234660|P""","""Butter""","""Dairy & Alternatives""","""Dairy""",30,10,[],"""Refrigerated""","""unit""","""1176804|S""",4,"""Daily Dairy""",55,85,false,false,"""Weekday""",false,"""High""",111,1.375948,1.421898,195,280,216
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1202127|P""","""Pomegranate""","""Fresh Foods""","""Fruits""",21,7,"[""September"", ""October"", … ""December""]","""Room Temperature""","""unit""","""1211763|S""",4,"""Tropical Fruits Ltd.""",350,80,true,false,"""Weekday""",false,"""High""",168,19.866171,1.963927,439,519,500


### Simulate Automated Purchase Orders

In [89]:
# Generate automated purchase orders driven by stock thresholds, minimum order quantities (MOQ), and demand forecasts
data_purchase_lazy = create_data_functions.create_purchase_order_polars(data_stock_dist_lazy)

### Standardize Final Columns Schema

In [90]:
# Define the final canonical column schema order for downstream modeling and storage compatibility
standardized_columns = ["order_purchase_date", "received_date", "product_id", "product", "category", "sub_category", "sales_demand", "sales_volume", "seasonality", "storage_recommendation", "unit_of_measurement", "shelf_life_days", "maximum_days_on_sale", "supplier_id", "supplier", "supplier_rating", "distance_km", "moq", "delivery_days", "transit_time", "in_season", "is_holiday", "day_classification", "is_weekend", "min_stock", "max_stock", "stock_quantity", "temperature_classification", "precipitation_classification", "wind_classification", "weather_severity"]

# Project the purchase lazy frame down to the definitive standardized column layout
std_df = data_purchase_lazy.select(pl.col(standardized_columns))

### Quantization

In [91]:
# View numeric columns
std_df.select(cs.numeric()).show()

sales_volume,shelf_life_days,maximum_days_on_sale,supplier_rating,distance_km,moq,delivery_days,transit_time,min_stock,max_stock,stock_quantity
i64,i64,i64,u8,i64,i64,f64,f64,i32,i32,i32
164,7,3,4,85,100,0.605329,2.414875,269,369,292
108,14,5,5,95,40,0.931616,2.768556,237,277,275
163,21,7,4,85,100,1.413887,2.173746,269,369,282
111,30,10,4,55,85,1.421898,1.375948,195,280,216
168,21,7,4,350,80,1.963927,19.866171,439,519,500


In [92]:
# Cast selected columns to smaller integer and float types 
# (UInt16 and Float16) to optimize memory usage and performance
qtz_df = std_df.with_columns(
    (pl.col("sales_volume")).cast(pl.UInt16),
    (pl.col("shelf_life_days")).cast(pl.UInt16),
    (pl.col("maximum_days_on_sale")).cast(pl.UInt16),
    (pl.col("distance_km")).cast(pl.UInt16),
    (pl.col("moq")).cast(pl.UInt16),
    (pl.col("delivery_days")).cast(pl.Float16),
    (pl.col("transit_time")).cast(pl.Float16),
    (pl.col("min_stock")).cast(pl.UInt16),
    (pl.col("max_stock")).cast(pl.UInt16),
    (pl.col("stock_quantity")).cast(pl.UInt16)
)

In [93]:
# Materialize and collect the final supply chain dataset into memory
qtz_df.collect()

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,str,str,str,str,str,u16,list[str],str,str,u16,u16,str,str,u8,u16,u16,f16,f16,bool,bool,str,bool,u16,u16,u16,str,str,str,str
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1113278|P""","""Carrot""","""Fresh Foods""","""Vegetables""","""High""",163,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""",21,7,"""1194877|S""","""ValleyFresh Farms""",4,85,100,1.4140625,2.173828,true,false,"""Weekday""",false,269,369,282,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-05,2022-12-09,"""1234660|P""","""Butter""","""Dairy & Alternatives""","""Dairy""","""High""",111,[],"""Refrigerated""","""unit""",30,10,"""1176804|S""","""Daily Dairy""",4,55,85,1.421875,1.375977,false,false,"""Weekday""",false,195,280,216,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1202127|P""","""Pomegranate""","""Fresh Foods""","""Fruits""","""High""",168,"[""September"", ""October"", … ""December""]","""Room Temperature""","""unit""",21,7,"""1211763|S""","""Tropical Fruits Ltd.""",4,350,80,1.963867,19.859375,true,false,"""Weekday""",false,439,519,500,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-19,2025-09-22,"""1230940|P""","""Haddock""","""Fresh Foods""","""Seafood""","""Normal""",6,[],"""Refrigerated""","""lb""",2,1,"""1891168|S""","""OceanHarvest Seafood""",2,180,40,0.788086,3.341797,false,false,"""Weekday""",false,23,63,118,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-18,2025-09-22,"""1385860|P""","""Granola Bars""","""Pantry""","""Snacks""","""Normal""",78,[],"""Room Temperature""","""unit""",90,30,"""1783257|S""","""SnackTime Distributors""",5,80,110,1.323242,2.681641,false,false,"""Weekday""",false,203,313,0,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-20,2025-09-22,"""1543068|P""","""Rye Bread""","""Bakery""","""Bread""","""Normal""",109,[],"""Room Temperature""","""unit""",5,2,"""1322480|S""","""Bakery Fresh Co.""",2,45,75,1.256836,1.758789,false,false,"""Weekday""",false,193,268,252,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""


# Save the DataFrame as a Parquet file using the fastparquet engine

In [94]:
# Save the DataFrame as a Parquet file using the fastparquet engine
# qtz_df.collect().write_parquet(file=processed_path / "grocery_data_pl.parquet")
qtz_df.collect().write_parquet(file=processed_path / "1M_grocery_data_pl.parquet")

# Performance

In [95]:
end = time.perf_counter()
end_CPU = time.process_time()

In [96]:
executionTime = processed_path / "execution.csv"

exec_time = {"polars_real": end - start, "polars_cpu": end_CPU - start_CPU}

with open(executionTime, "a", newline="") as f:
    writer = csv.writer(f)
    if f.tell() == 0:
        writer.writerow(["Execution", "Time(s)"])
    for item in exec_time.items():
        writer.writerow([item[0], item[1]])


print("Pandas Time execution all time and only CPU")
print(f"Real Time:  {list(exec_time.values())[0]}")
print(f"CPU Time:  {list(exec_time.values())[1]}")

Pandas Time execution all time and only CPU
Real Time:  17.42360344700046
CPU Time:  25.813384528999997
